# Kununu Webscraping

Scrapes employee reviews from Kununu for a set of German companies using Playwright.  
Results are saved to `reviews_json/` and loaded into the `REVIEWS_GER` table in `PJDB.db`.

In [ ]:
#%pip install playwright beautifulsoup4 lxml python-dotenv

In [1]:
import re, hashlib, math
from bs4 import BeautifulSoup
from pathlib import Path

KUNUNU_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
}

# Maps every German h4 category label on Kununu → English field name
CATEGORY_MAP = {
    "Gut am Arbeitgeber finde ich":      "pros",
    "Schlecht am Arbeitgeber finde ich": "cons",
    "Verbesserungsvorschläge":           "suggestions",
    "Arbeitsatmosphäre":                 "atmosphere",
    "Image":                             "image",
    "Work-Life-Balance":                 "work_life_balance",
    "Karriere/Weiterbildung":            "career",
    "Gehalt/Sozialleistungen":           "salary",
    "Kollegenzusammenhalt":              "colleagues",
    "Vorgesetztenverhalten":             "management",
    "Arbeitsbedingungen":                "working_conditions",
    "Gleichberechtigung":                "equal_rights",
    "Interessante Aufgaben":             "interesting_tasks",
    "Kommunikation":                     "communication",
    "Umgang mit älteren Kollegen":       "senior_colleagues",
    "Umwelt-/Sozialbewusstsein":         "environment_social",
}

def build_url(slug: str, page: int) -> str:
    return f"https://www.kununu.com/de/{slug}/kommentare/{page}"

def detect_total_reviews(html: str) -> int | None:
    for pattern in [
        r'"Basierend auf ([\d.]+) Erfahrungen',
        r'([\d.]+)\s+Erfahrungen?\b',
        r'([\d.]+)\s+Bewertungen?\b',
    ]:
        m = re.search(pattern, html)
        if m:
            return int(m.group(1).replace(".", ""))
    return None

def parse_reviews(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    articles = soup.find_all("article", attrs={
        "data-testid": re.compile(r"^review-[0-9a-f-]{36}$")
    })
    reviews = []
    for a in articles:
        rating_el = a.find(attrs={"data-testid": "review-rating"})
        rating = rating_el.get_text(strip=True).replace(",", ".") if rating_el else None

        rec_el = a.find(attrs={"data-testid": "review-recommendation"})
        recommendation = rec_el.get_text(strip=True) if rec_el else None

        time_el = a.find("time")
        date = time_el.get("datetime", "").split("T")[0] if time_el else None

        b_el = a.find("b")
        position = b_el.get_text(strip=True) if b_el else None

        h3_el = a.find("h3")
        headline = h3_el.get_text(strip=True) if h3_el else None

        r = {
            "date":           date,
            "rating":         rating,
            "recommendation": recommendation,
            "position":       position,
            "headline":       headline,
            **{field: None for field in CATEGORY_MAP.values()},
        }

        for h4 in a.find_all("h4"):
            label = h4.get_text(strip=True)
            if label in CATEGORY_MAP:
                p = h4.find_next_sibling("p")
                if p:
                    r[CATEGORY_MAP[label]] = " ".join(p.get_text(separator=" ", strip=True).split())

        if any(v for v in r.values()):
            reviews.append(r)
    return reviews

def _fingerprint(r: dict) -> str:
    key = (r.get("pros") or "") + (r.get("cons") or "") + (r.get("position") or "") + (r.get("date") or "")
    return hashlib.md5(key.encode()).hexdigest()

In [ ]:
# test to verify playwright + selectors work
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
import time, random

async def quick_test(slug="siemens", page=1):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent=KUNUNU_HEADERS["User-Agent"],
            locale="de-DE",
            extra_http_headers={"Accept-Language": KUNUNU_HEADERS["Accept-Language"]},
        )
        pg = await ctx.new_page()
        await pg.goto(build_url(slug, page), wait_until="domcontentloaded", timeout=30_000)
        await pg.wait_for_selector("article[data-testid^='review-']", timeout=15_000)
        html = await pg.content()
        await browser.close()

    reviews = parse_reviews(html)
    print(f"Reviews found: {len(reviews)}")
    if reviews:
        print("Sample review:")
        for k, v in reviews[0].items():
            if v:
                print(f"  {k}: {str(v)[:80]}")
    return reviews

await quick_test("siemens")

Reviews found: 10
Sample review:
  date: 2026-06-04
  rating: 4.9
  recommendation: Empfohlen
  position: Werkstudent/in
  headline: Top Arbeitgeber für Werkstudenten
  atmosphere: Sehr internationales Team mit vielen offenen Kollegen.
  image: Sehr positiv
  work_life_balance: 35 Stundenwoche lässt viel Ausgleich neben der Arbeit zu.
  career: Viele Möglichkeiten und Förderung der internen Weiterbildung.
  salary: Mit eines der besten Gehälter für Werkstudenten, inklusive Urlaubsgeld, etc.
  colleagues: Sehr gute Teamarbeit mit Kollegen.
  management: Vorgesetzte reden auf Augenhöhe, generell keine Starken Hierarchien sichtbar, wa
  working_conditions: Sehr angenehm als Werkstudent.
  equal_rights: keine Benachteiligung bestimmter Personen Gruppen erlebt.
  interesting_tasks: Sehr abwechslungsreiche Aufgaben, da immer unterschiedliche Projekte laufen.
  communication: Sehr gute Kommunikation über aktuelle Ereignisse und Updates vom Management.
  senior_colleagues: Ältere Kollegen werd

[{'date': '2026-06-04',
  'rating': '4.9',
  'recommendation': 'Empfohlen',
  'position': 'Werkstudent/in',
  'headline': 'Top Arbeitgeber für Werkstudenten',
  'pros': None,
  'cons': None,
  'suggestions': None,
  'atmosphere': 'Sehr internationales Team mit vielen offenen Kollegen.',
  'image': 'Sehr positiv',
  'work_life_balance': '35 Stundenwoche lässt viel Ausgleich neben der Arbeit zu.',
  'career': 'Viele Möglichkeiten und Förderung der internen Weiterbildung.',
  'salary': 'Mit eines der besten Gehälter für Werkstudenten, inklusive Urlaubsgeld, etc.',
  'colleagues': 'Sehr gute Teamarbeit mit Kollegen.',
  'management': 'Vorgesetzte reden auf Augenhöhe, generell keine Starken Hierarchien sichtbar, was mich sehr überrascht hat.',
  'working_conditions': 'Sehr angenehm als Werkstudent.',
  'equal_rights': 'keine Benachteiligung bestimmter Personen Gruppen erlebt.',
  'interesting_tasks': 'Sehr abwechslungsreiche Aufgaben, da immer unterschiedliche Projekte laufen.',
  'communic

In [1]:
import os, asyncio, math, json, random
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout


async def scrape_all_reviews(
    company_name: str,
    slug: str,
    max_reviews: int | None = None,
    delay: float = 2.5,
    checkpoint_file: str | None = None,
    empty_page_retries: int = 2,
) -> list[dict]:
   
    all_reviews: list[dict] = []
    start_page = 1

    if checkpoint_file and os.path.exists(checkpoint_file):
        with open(checkpoint_file, encoding="utf-8") as f:
            ckpt = json.load(f)
        all_reviews = ckpt.get("reviews", [])
        start_page  = ckpt.get("next_page", 1)
        print(f"Resuming from page {start_page} — {len(all_reviews)} reviews already collected.")

    consecutive_empty = 0
    total_known = None
    page_num = start_page

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent=KUNUNU_HEADERS["User-Agent"],
            locale="de-DE",
            extra_http_headers={"Accept-Language": KUNUNU_HEADERS["Accept-Language"]},
        )
        pg = await ctx.new_page()

        await pg.goto("https://www.kununu.com/de", wait_until="domcontentloaded", timeout=30_000)
        await asyncio.sleep(random.uniform(1.5, 2.5))

        while True:
            if max_reviews is not None and len(all_reviews) >= max_reviews:
                print(f"Reached target of {max_reviews} reviews.")
                break

            url = build_url(slug, page_num)
            try:
                await pg.goto(url, wait_until="domcontentloaded", timeout=30_000)
                await pg.wait_for_selector("article[data-testid^='review-']", timeout=15_000)
            except PlaywrightTimeout:
                consecutive_empty += 1
                print(f"Page {page_num}: timeout ({consecutive_empty}/{empty_page_retries + 1})")
                if consecutive_empty > empty_page_retries:
                    print("Too many timeouts — stopping.")
                    break
                await asyncio.sleep(4)
                continue

            consecutive_empty = 0
            html = await pg.content()

            if total_known is None:
                total_known = detect_total_reviews(html)
                if total_known:
                    target = min(total_known, max_reviews) if max_reviews else total_known
                    print(f"Detected ~{total_known:,} total reviews "
                          f"→ targeting {target:,} (~{math.ceil(target / 10)} pages)")

            page_reviews = parse_reviews(html)
            if not page_reviews:
                consecutive_empty += 1
                if consecutive_empty > empty_page_retries:
                    print("No reviews on consecutive pages — done.")
                    break
            else:
                consecutive_empty = 0
                all_reviews.extend(page_reviews)
                print(f"Page {page_num:>4}: {len(page_reviews):>3} scraped  |  total: {len(all_reviews):,}")

            if checkpoint_file:
                with open(checkpoint_file, "w", encoding="utf-8") as f:
                    json.dump({"reviews": all_reviews, "next_page": page_num + 1},
                              f, ensure_ascii=False, indent=2)

            page_num += 1
            await asyncio.sleep(delay + random.random() * 0.8)

        await browser.close()

    return all_reviews[:max_reviews] if max_reviews else all_reviews

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("reviews_json")
OUTPUT_DIR.mkdir(exist_ok=True)

companies = [
    # Tech / IT
    ("SAP",              "sap"),
    ("IBM",              "ibm"),

    # Automotive & Mobility
    ("BMW",              "bmw-ag"),
    ("Mercedes-Benz",    "mercedes-benz"),
    ("Volkswagen",       "volkswagen"),
    ("Continental",      "continental"),
    ("Daimler-Truck",    "daimler-truck"),

    # Industrial & Engineering
    ("Siemens",          "siemens"),
    ("Siemens-Energy",   "siemens-energy-global"),
    ("Airbus",           "airbus-group"),
    ("Rheinmetall",      "rheinmetall-group"),
    ("MTU",              "mtu"),
    ("Infineon",         "infineon-technologies"),

    # Chemicals / Pharma / Life Sciences
    ("BASF",             "basf"),
    ("Bayer",            "bayer"),
    ("Merck",            "merck"),
    ("Fresenius",        "fresenius-se"),
    ("Sartorius",        "sartorius"),
    ("Symrise",          "symrise"),

    # Consumer & Retail
    ("Adidas",           "adidas"),
    ("Beiersdorf",       "beiersdorf"),
    ("Zalando",          "zalando"),

    # Finance / Insurance
    ("Allianz",          "allianz"),
    ("Munich-Re",        "munichre"),
    ("Deutsche-Bank",    "deutsche-bank"),
    ("Commerzbank",      "commerzbank"),

    # Energy / Utilities
    ("E-ON",             "eon"),
    ("RWE",              "rwe"),

    # Telecom & Logistics
    ("Deutsche-Telekom", "deutsche-telekom"),
    ("Deutsche-Post",    "deutsche-post"),

    # Real Estate
    ("Vonovia",          "vonovia"),

    # Large employers not in the DAX
    ("Lufthansa",        "lufthansa"),
    ("Deutsche-Bahn",    "deutsche-bahn"),
    ("Thyssenkrupp",     "thyssenkrupp"),
    ("Lidl",             "lidl"),
    ("Rewe",             "rewe"),
    ("Kaufland",         "kaufland"),
]

MAX_REVIEWS = 500
all_results = {}

for company_name, slug in companies:
    out_file  = OUTPUT_DIR / f"{company_name.lower()}_reviews.json"
    ckpt_file = OUTPUT_DIR / f"{company_name.lower()}_checkpoint.json"

    print(f"\n{'='*60}")
    print(f"Scraping: {company_name}  (slug: {slug})")
    print(f"{'='*60}")

    reviews = await scrape_all_reviews(
        company_name=company_name,
        slug=slug,
        max_reviews=MAX_REVIEWS,
        delay=2.5,
        checkpoint_file=str(ckpt_file),
    )

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)

    all_results[company_name] = reviews
    print(f"Saved {len(reviews):,} reviews → {out_file}\n")

### Additional Companies

Second scraping pass targeting lower-rated employers and additional industries to balance the sample.

In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path("reviews_json")
OUTPUT_DIR.mkdir(exist_ok=True)

low_rating_companies = [
    # Logistics
    ("Hermes",      "hermes-germany"),
    ("GLS",         "gls-germany"),

    # Discount 
    ("Aldi-Nord",   "aldi-nord"),
    ("Penny",       "penny-markt"),
    ("Netto",       "netto-marken-discount"),

    # Fast Food
    ("McDonalds",   "mcdonalds-deutschland"),
    ("Burger-King", "burger-king-deutschland"),

    # Staffing
    ("Randstad",    "randstad"),
    ("Arvato",      "arvato"),
]

MAX_REVIEWS = 500

for company_name, slug in low_rating_companies:
    out_file  = OUTPUT_DIR / f"{company_name.lower()}_reviews.json"
    ckpt_file = OUTPUT_DIR / f"{company_name.lower()}_checkpoint.json"

    if out_file.exists():
        existing = json.loads(out_file.read_text(encoding="utf-8"))
        print(f"  {company_name}: already scraped ({len(existing):,} reviews) — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Scraping: {company_name}  (slug: {slug})")
    print(f"{'='*60}")

    reviews = await scrape_all_reviews(
        company_name=company_name,
        slug=slug,
        max_reviews=MAX_REVIEWS,
        delay=2.5,
        checkpoint_file=str(ckpt_file),
    )

    if reviews:
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(reviews, f, ensure_ascii=False, indent=2)
        print(f"{len(reviews):,} reviews saved → {out_file}")
    else:
        print(f"No reviews found for {company_name} — check slug: kununu.com/de/{slug}")